# Introduction
This notebook demonstrates how to set up and run a quantized version of the Llama-3-8B model. We will begin with some basic setup and then proceed to load and use the model.

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 39.3936


In [2]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface/hub/"
print(f"Setting cache path to {CACHE_PATH}")

os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH
os.environ["HUGGINGFACE_HUB_CACHE"] = CACHE_PATH
os.environ["HUGGINGFACE_ASSETS_CACHE"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Code formatting and linting

# !black notebooks/Llama-3-8B-quant.ipynb
# !pylint notebooks/Llama-3-8B-quant.ipynb

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface/hub/
MemTotal: 1007.71 GB
MemFree: 919.79 GB
MemAvailable: 980.80 GB
Free GPU Memory (GB): 39.3936

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA A100-PCIE-40GB

################################
Authentication with Hugging Face...
################################



/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/hub/token
Login successful
Successfully authenticated with the Hugging Face API.

################################
Setting up GPU memory usage list...
################################



## 2. Loading Model

In [4]:
from transformers import AutoTokenizer

# model_name = "meta-llama/Meta-Llama-3-8B"
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda"

# model_name = "EleutherAI/gpt-neo-125m"  # Lightweight model for debugging purposes
model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.
# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.
# model_name = "openai-community/gpt2-large"

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
# TODO: Check why dtype = auto solved the problem
# TODO: what is the default value of torch_dtype -> look in the githubb documentation
# Always use "auto"
model.NAME = model_name

tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

if tokenizer.model_max_length > 1e6:
  print(f"Tokenizer model max length reduced from {tokenizer.model_max_length} to 2048 to fit in memory")
  tokenizer.model_max_length = 2048

!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
print(f"Loaded model {model_name} with the following configuration:")
print(f"- model max length: {tokenizer.model_max_length}")
print(f"- dtype: {model.dtype}")
print(f"- device: {model.device}")
print(f"- parameters: {(lambda p: f'{p / 1e9:.1f}B' if p > 1e9 else (f'{p / 1e6:.1f}M' if p > 1e6 else str(p)))(model.num_parameters())}")
print(f"- memory footprint: {model.get_memory_footprint() / (1024 ** 3):.2f} GB")
print(f"- vocabulary size: {tokenizer.vocab_size}")
print(f"- padding token ID: {tokenizer.pad_token_id}")
print(f"- special tokens: {tokenizer.special_tokens_map}")

from src.evaluations.evaluate_memory import record_gpu_memory
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Load model")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.15s/it]

Tokenizer model max length reduced from 1000000000000000019884624838656 to 2048 to fit in memory


Free GPU Memory (GB): 23.8965
Loaded model meta-llama/Meta-Llama-3-8B with the following configuration:
- model max length: 2048
- dtype: torch.bfloat16
- device: cuda:0
- parameters: 8.0B
- memory footprint: 14.96 GB
- vocabulary size: 128000
- padding token ID: 128001
- special tokens: {'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>', 'pad_token': '<|end_of_text|>'}
Free GPU Memory (GB): 23.8965. Context: Load model.


In [ ]:
# Example inference

from transformers import AutoTokenizer
import transformers 
import torch

tokenizer = AutoTokenizer.from_pretrained(model_name)
pipeline = transformers.pipeline(
    "text-generation",
    model=model_name,
    torch_dtype="auto",
    device_map="auto",
)

prompt = "What famous tower is in Paris?"
formatted_prompt = (
    f"### Human: {prompt}### Assistant:"
)

sequences = pipeline(
    formatted_prompt,
    do_sample=True,
    top_k=50,
    top_p = 0.7,
    num_return_sequences=1,
    repetition_penalty=1.1,
    max_new_tokens=500,
)
for seq in sequences:
    print(f"Result: {seq['generated_text']}")


## 3. Loading Datasets

### 3.1. WikiText

In [4]:
# Initialize the datamodule
import os
from src.data.WikiTextDataModule import WikiTextDataModule

print("\n################################")
print("Setting up WikiTextDataModule...")
print("################################\n")

wikitext_data_module = WikiTextDataModule(
  directory_dataset=os.getcwd(),
  batch_size=1,
  sequence_length=256,
  tokenizer_name=model_name,
  seed=3,
)

wikitext_dataloader = wikitext_data_module.test_dataloader()

print("\n################################")
print("Printing properties of WikiTextDataModule...")
print("################################\n")

# Print properties
print(f"Length of train dataset: {len(wikitext_data_module.train_dataset)}")
print(f"Length of validation dataset: {len(wikitext_data_module.val_dataset)}")
print(f"Length of test dataset: {len(wikitext_data_module.test_dataset)}")

print("\nTotal number of tokens in each dataset:")
print(f"Train dataset: {sum([len(data_string) for data_string in wikitext_data_module.train_dataset['text']])}")
print(f"Validation dataset: {sum([len(data_string) for data_string in wikitext_data_module.val_dataset['text']])}")
print(f"Test dataset: {sum([len(data_string) for data_string in wikitext_data_module.test_dataset['text']])}")

total_string = "".join([data_string for data_string in wikitext_data_module.val_dataset['text']])
total_string_len = len(total_string)
tokenized_string = tokenizer.encode(total_string, return_tensors="pt")

print(f"\nLength of total validation dataset (characters): {total_string_len}")
print(f"Length of tokenized validation dataset (tokens): {len(tokenized_string[0])}")
print(f"Tokenizer compression rate: {(100 * len(tokenized_string[0]) / total_string_len):.2f}%")

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset
dataset_size = len(wikitext_dataloader)
print(f"\nNumber of batches in validation dataloader: {dataset_size}")

for i, (data, target) in enumerate(wikitext_dataloader):
    if i < 1:
        print(f"\nBatch {i + 1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}...")  # Print the first 500 characters
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")


################################
Setting up WikiTextDataModule...
################################



Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.



################################
Printing properties of WikiTextDataModule...
################################

Length of train dataset: 36718
Length of validation dataset: 3760
Length of test dataset: 4358

Total number of tokens in each dataset:
Train dataset: 10892990
Validation dataset: 1142150
Test dataset: 1285622

Length of total validation dataset (characters): 1142150
Length of tokenized validation dataset (tokens): 252726
Tokenizer compression rate: 22.13%

Number of batches in validation dataloader: 564

Batch 1:
  Original Text:  may choose any order to play the overland sections, various obstacles prevent the player from entering the dungeons outside of a specific order. In addition to this, some levels provide the player with vital clues which solve puzzles needed to progress in later sections. Once Donald has completed the overland section of an area, he may leave by calling his nephews'biplane, and will return to the dungeon entrance of that area if the player chooses 

### 3.2. OpenAssistant

In [ ]:
# Initialize the datamodule
import os
from src.data.OpenAssistantDataModule import OpenAssistantDataModule

# Data Module
oasst_data_module = OpenAssistantDataModule(
  directory_dataset=os.getcwd(),
  batch_size=1,
  sequence_length=2048,
  tokenizer_name=model_name,
  seed=1
)

# Data Loader
# oasst_dataloader = oasst_data_module.train_dataloader()
oasst_dataloader = oasst_data_module.val_dataloader()

print(f"Length of datasets:", len(oasst_data_module.train_dataset), len(oasst_data_module.val_dataset))

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset

oasst_dataset_size = len(oasst_dataloader)
print(f"Number of batches in train_dataloader: {dataset_size}")
for i, (data, target) in enumerate(oasst_dataloader):
    if i < 2:
        print(f"Batch {i+1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}")
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")

## 4. Quantization

### 4.4 Quanto

### 4.4.1 HuggingFace Integration - Won't work!!!

In [5]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, QuantoConfig

from src import MODEL_SAVE_PATH

model_name = "meta-llama/Meta-Llama-3-8B"
tokenizer_instance = AutoTokenizer.from_pretrained(model_name)
tokenizer_instance.pad_token_id = tokenizer_instance.eos_token_id

quantization_config = QuantoConfig(weights="int8", activations="int8")

quanto_model_1 = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    torch_dtype="auto",
    device_map="cuda"
)

# Set the model's PATH and NAME attributes
quanto_model_1_name = f"{model_name.split('/')[1]}-QUANTO-1"
quanto_model_1_path = os.path.join(MODEL_SAVE_PATH, quanto_model_1_name)
quanto_model_1.PATH = quanto_model_1_path
quanto_model_1.NAME = quanto_model_1_name

# Save the quantized model using `safetensors`
from safetensors.torch import save_file

# Save the model state dictionary
safe_file_path = f"{quanto_model_1_path}.safetensors"
save_file(quanto_model_1.state_dict(), safe_file_path)

# Save the quantization map to a JSON file
import json
from optimum.quanto import quantization_map

quantization_map_path = f"{quanto_model_1_path}_quantization_map.json"
with open(quantization_map_path, 'w') as f:
    json.dump(quantization_map(quanto_model_1), f)

print(f"Quantized model saved at {safe_file_path}")
print(f"Quantization map saved at {quantization_map_path}")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


ValueError: We don't support quantizing the activations with transformers library.Use quanto library for more complex use cases such as activations quantization, calibration and quantization aware training.

### 4.4.2 Optimum Quanto - No Finetuning

In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from optimum.quanto import quantize, freeze, qint8, qfloat8, safe_save
from src import MODEL_SAVE_PATH

# Load the model and tokenizer
quanto_model_2 = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
tokenizer_instance = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
tokenizer_instance.pad_token_id = tokenizer_instance.eos_token_id

# Quantize the model
quantize(quanto_model_2, weights=qint8, activations=qfloat8)

# Freeze the model to convert weights to integers
freeze(quanto_model_2)

quanto_model_2_name = f"{model_name.split('/')[1]}-QUANTO-2"
quanto_model_2_path = os.path.join(MODEL_SAVE_PATH, quanto_model_2_name)
quanto_model_2.PATH = quanto_model_2_path
quanto_model_2.NAME = quanto_model_2_name

# Save the quantized model
safe_save(quanto_model_2.state_dict(), f"{quanto_model_2_path}.safetensors")

RuntimeError: Failed to import transformers.models.llama.modeling_llama because of the following error (look up to see its traceback):
This is not allowed since there's already a kernel registered from python overriding dqmm's behavior for CPU dispatch key and quanto_ext namespace.

### 4.4.3 Optimum Quanto - Calibration

In [5]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Free GPU Memory (GB): 31.1895


In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from optimum.quanto import Calibration, quantize, freeze, qint8, safe_save
from src import MODEL_SAVE_PATH

# Load the model and tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
quanto_model_3 = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
tokenizer_instance = AutoTokenizer.from_pretrained(model_name)
tokenizer_instance.pad_token_id = tokenizer_instance.eos_token_id

# Load the calibration dataset
cal_dataset = wikitext_data_module.val_dataset

# Quantize the model
quantize(quanto_model_3, weights=qint8, activations=qfloat8)
# qfloat8 vs qint8 -> weight quantization is easier than activations -> maybe not quantizing activations at all?

# Calibrate the model
cal_samples = 20  # Set to lower values for debugging
with Calibration(streamline=True, debug=False):
    quanto_model_3.eval()
    total = 0
    for batch in cal_dataset.iter(batch_size=1):
        print(f"Processing batch {total + 1}/{cal_samples}")
        inputs = tokenizer_instance(batch["text"], return_tensors="pt", padding=True).to(device)
        input_ids = inputs.input_ids
        attention_mask = inputs.attention_mask
        quanto_model_3(input_ids, attention_mask=attention_mask)
        total += input_ids.size(0)
        if total >= cal_samples:
            break

# Freeze the model to convert weights to integers
freeze(quanto_model_3)

quanto_model_3_name = f"{model_name.split('/')[1]}-QUANTO-3"
quanto_model_3_path = os.path.join(MODEL_SAVE_PATH, quanto_model_3_name)
quanto_model_3.PATH = quanto_model_3_path
quanto_model_3.NAME = quanto_model_3_name

# Save the quantized model
safe_save(quanto_model_3.state_dict(), f"{quanto_model_3_path}.safetensors")

Processing batch 1/128
Processing batch 2/128
Processing batch 3/128
Processing batch 4/128
Processing batch 5/128
Processing batch 6/128
Processing batch 7/128
Processing batch 8/128
Processing batch 9/128
Processing batch 10/128
Processing batch 11/128
Processing batch 12/128
Processing batch 13/128
Processing batch 14/128
Processing batch 15/128
Processing batch 16/128
Processing batch 17/128
Processing batch 18/128
Processing batch 19/128
Processing batch 20/128
Processing batch 21/128
Processing batch 22/128
Processing batch 23/128
Processing batch 24/128
Processing batch 25/128
Processing batch 26/128
Processing batch 27/128
Processing batch 28/128
Processing batch 29/128
Processing batch 30/128
Processing batch 31/128
Processing batch 32/128
Processing batch 33/128
Processing batch 34/128
Processing batch 35/128
Processing batch 36/128
Processing batch 37/128
Processing batch 38/128
Processing batch 39/128
Processing batch 40/128
Processing batch 41/128
Processing batch 42/128
P

### 4.4.4 Optimum Quanto - QAT

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Free GPU Memory (GB): 36.8418


In [6]:
model_name

'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from optimum.quanto import quantize, freeze, qint8, safe_save, QTensor
from torch.optim import Adam
from src import MODEL_SAVE_PATH

# Load the model and tokenizer
print("Loading the model and tokenizer...")
quanto_model_4 = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map=device)
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
tokenizer.pad_token_id = tokenizer.eos_token_id

# Load the calibration dataset
cal_dataset = wikitext_data_module.val_dataset

# Quantize the model
print("Quantizing the model...")
quantize(quanto_model_4, weights=qint8, activations=qint8)

# Quantization-Aware Training (QAT)
print("Quantization-Aware Training (QAT)...")
train_samples = 5
train_dataloader = wikitext_data_module.test_dataloader()
quanto_model_4.train()
optimizer = Adam(quanto_model_4.parameters(), lr=1e-4)
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

n_epochs = 3  # Set lower values for debugging
for epoch in range(n_epochs):
    for batch_idx, (data, target) in enumerate(train_dataloader):
        print(f"Processing batch {batch_idx + 1}/{train_samples}")
        print("Memory before model:")
        !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
        if batch_idx >= train_samples:
            break
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            output = quanto_model_4(data)
            print("Memory after model:")
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
            logits = output.logits
            if isinstance(logits, QTensor):
                print("Dequantizing logits...")
                logits = logits.dequantize()
            loss = torch.nn.functional.nll_loss(logits.view(1, logits.shape[2], logits.shape[1]), target)
            loss.backward()
            optimizer.step()
            
# Freeze the model to convert weights to integers
print("Freezing the model...")
freeze(quanto_model_4)

quanto_model_4_name = f"{model_name.split('/')[1]}-QUANTO-4"
quanto_model_4_path = os.path.join(MODEL_SAVE_PATH, quanto_model_4_name)
quanto_model_4.PATH = quanto_model_4_path
quanto_model_4.NAME = quanto_model_4_name

# Save the quantized model
print("Saving the quantized model...")
safe_save(quanto_model_4.state_dict(), f"{quanto_model_4_path}.safetensors")

Loading the model and tokenizer...


Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.18s/it]
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Quantizing the model...
Quantization-Aware Training (QAT)...
Free GPU Memory (GB): 22.7441
Processing batch 1/5
Memory before model:
Free GPU Memory (GB): 22.7441


OutOfMemoryError: CUDA out of memory. Tried to allocate 112.00 MiB. GPU 

In [6]:
loss

tensor(0.5801, device='cuda:0')

In [ ]:
# Freeze the model to convert weights to integers
print("Freezing the model...")
freeze(quanto_model_4)

quanto_model_4_name = f"{model_name.split('/')[1]}-QUANTO-4"
quanto_model_4_path = os.path.join(MODEL_SAVE_PATH, quanto_model_4_name)
quanto_model_4.PATH = quanto_model_4_path
quanto_model_4.NAME = quanto_model_4_name

# Save the quantized model
print("Saving the quantized model...")
safe_save(quanto_model_4.state_dict(), f"{quanto_model_4_path}.safetensors")

## 5. Evaluation

In [6]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Free GPU Memory (GB): 4.53906


### 5.1. Perplexity

In [7]:
quanto_model_2.to("cuda")

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): QLinear(in_features=2048, out_features=2048, bias=False)
          (k_proj): QLinear(in_features=2048, out_features=256, bias=False)
          (v_proj): QLinear(in_features=2048, out_features=256, bias=False)
          (o_proj): QLinear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): QLinear(in_features=2048, out_features=5632, bias=False)
          (up_proj): QLinear(in_features=2048, out_features=5632, bias=False)
          (down_proj): QLinear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNorm()
  )
  (lm_head

In [6]:
import torch
import torchmetrics
import tqdm
from torch.cuda.amp import autocast

# Evaluate Perplexity
print("\n################################")
print("Evaluating Perplexity...")
print("################################\n")

def evaluate_perplexity(model, dataloader, device="cuda", to_device=False):
    if isinstance(model, torch.nn.Module):
        model.eval()
        print(f"Model in evaluation mode. Device: {device}")
    metric = torchmetrics.text.Perplexity(ignore_index=-100).to(device)  # -100 is the padding token.

    for i, (x, y) in enumerate(dataloader):
        print(f"Processing batch {i}")
        x, y = x.to(device), y.to(device)
        
        with torch.no_grad():
            outputs = model(x)
            logits = outputs.logits
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
            
            # Metric on current batch
            perplexity = metric(logits.dequantize(), y)
            print(f"Perplexity: {perplexity:.2f}")

    # Metric on all batches using custom accumulation
    perplexity = metric.compute()
    print(f"\nFinal Perplexity (PPL): {perplexity:.3f}")
    return perplexity.item()

wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(quanto_model_2, wikitext_dataloader, device="cuda")
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'


################################
Evaluating Perplexity...
################################

Model in evaluation mode. Device: cuda
Processing batch 0


551.95s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Free GPU Memory (GB): 18.2109
Free GPU Memory (GB): 7.87402
Perplexity: 732169.31
Processing batch 1


577.90s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Free GPU Memory (GB): 18.1699
Free GPU Memory (GB): 7.87402
Perplexity: 978260.62
Processing batch 2


...

Free GPU Memory (GB): 18.1699
Free GPU Memory (GB): 7.87402
Perplexity: 724549.44
Processing batch 10


1850.81s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Free GPU Memory (GB): 18.1699
Free GPU Memory (GB): 7.87402
Perplexity: 343396.22
Processing batch 11


1856.20s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Free GPU Memory (GB): 18.1699
Free GPU Memory (GB): 7.87402


In [9]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Model in evaluation mode. Device: cuda
Processing batch 0
Free GPU Memory (GB): 3.92773
Perplexity: 5.55
Processing batch 1
Free GPU Memory (GB): 3.60156
Perplexity: 9.71
Processing batch 2
Free GPU Memory (GB): 3.63086
Perplexity: 12.43
Processing batch 3
Free GPU Memory (GB): 3.63477
Perplexity: 11.15
Processing batch 4
Free GPU Memory (GB): 3.63477
Perplexity: 5.95
Processing batch 5
Free GPU Memory (GB): 3.63477
Perplexity: 7.38
Processing batch 6
Free GPU Memory (GB): 3.60156
Perplexity: 5.51
Processing batch 7
Free GPU Memory (GB): 3.60156
Perplexity: 4.95
Processing batch 8
Free GPU Memory (GB): 3.60156
Perplexity: 7.85
Processing batch 9
Free GPU Memory (GB): 3.59375
Perplexity: 8.51
Processing batch 10
Free GPU Memory (GB): 3.63477
Perplexity: 8.89
Processing batch 11
Free GPU Memory (GB): 3.63477
Perplexity: 7.77
Processing batch 12
Free GPU Memory (GB): 3.63477
Perplexity: 6.97
Processing batch 13
Free GPU Memory (GB): 3.63477
Perplexity: 8.06
Processing batch 14
Free GPU Me

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(quantized_model, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model_same, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model_dynamic, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

In [ ]:
import numpy as np
lls = torch.tensor(lls)
print(stride)
print(lls/stride)
print(torch.exp(lls / (stride)))
print(torch.exp(lls.sum() / (31 * stride)))

ppls = [ppl for ppl in ppls]
print(ppls)

print(xs[2])
print(ys[2])
print(input_ids_list[2])
print(target_ids_list[2])

print(outputs[0])
print()

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [ ]:
evaluate_perplexity(model_bnb_8bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_perplexity(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_perplexity(awq_model, wikitext_dataloader, device="cuda")

In [ ]:
list_of_models = [model, model_bnb_8bit, model_bnb_4bit]
results = {}
for model in list_of_models:
  print(f"Perplexity for model {model.NAME}: {evaluate_perplexity(model_bnb_8bit, wikitext_data_module, device)}"

# Print perplexity results
#print(f"Perplexity (8-bit): {perplexity_8bit:.4f}")
#print(f"Perplexity (4-bit): {perplexity_4bit:.4f}")
print(f"Perplexity (Original): {perplexity_original:.4f}")

### 5.2. Brier Score

In [ ]:
import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast

class BrierScore:
    def __init__(self, device="cpu"):
        self.device = device
        self.reset()

    def reset(self):
        self.total_brier_score = 0.0
        self.num_batches = 0

    def update(self, probs, targets):
        brier_score = torch.mean((probs - targets) ** 2)
        self.total_brier_score += brier_score.item()
        self.num_batches += 1

    def compute(self):
        if self.num_batches == 0:
            return 0.0
        return self.total_brier_score / self.num_batches

def evaluate_brier_score(model, dataloader, device="cuda", to_device=False):
    if to_device:
        model.to(device)

    if isinstance(model, torch.nn.Module):
        model.eval()

    print(f"Model in evaluation mode. Device: {device}")
    
    # Initialize BrierScore metric
    metric = BrierScore(device=device)
    
    for i, (x, y) in enumerate(dataloader):
        if i > 10:
            break
        print(f"Processing batch {i}")
        x, y = x.to(device), y.to(device)

        with torch.no_grad() and autocast():
            outputs = model(x)
            logits = outputs.logits
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

            # Shift logits and target_ids to the left by 1 for calculating the Brier score
            shifted_logits = logits[:, :-1].contiguous()
            shifted_target_ids = x[:, 1:].contiguous()

            # Flatten the logits and target_ids for calculation
            shifted_logits = shifted_logits.view(-1, shifted_logits.size(-1))
            shifted_target_ids = shifted_target_ids.view(-1)

            # Filter out the -100 targets
            valid_indices = shifted_target_ids != -100
            valid_logits = shifted_logits[valid_indices]
            valid_target_ids = shifted_target_ids[valid_indices]

            # Get the probabilities
            probs = F.softmax(valid_logits, dim=-1)

            # Create one-hot target vectors
            targets = F.one_hot(valid_target_ids, num_classes=probs.size(-1)).float()

            # Update the metric with the current batch's results
            metric.update(probs, targets)

    # Compute the final Brier score across all batches
    avg_brier_score = metric.compute()
    print(f"Final Brier Score: {avg_brier_score:.10f}")

    return avg_brier_score

# Assuming wikitext_data_module and model are defined elsewhere
wikitext_dataloader = wikitext_data_module.test_dataloader()
final_brier_score = evaluate_brier_score(model, wikitext_dataloader, device=device)
print(f"\nFinal Brier Score: {final_brier_score:.10f}")

In [ ]:
evaluate_brier_score(model, tokenizer, wikitext_dataloader, factor=100, device=device)

In [ ]:
evaluate_brier_score(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)

## 6. Plotting activation values

In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
import os
import numpy as np
from optimum.quanto import quantize, freeze, qint8, qfloat8, safe_save

# Helper functions to register hooks and extract activations
def get_activation(name, activations):
    def hook(model, input, output):
        if isinstance(output, tuple):
            output = output[0]
        activations[name] = output.detach()
    return hook

def plot_activations_pair(layer_name, activations_original, activations_quanto):
    activation_values_original = activations_original[layer_name].cpu().numpy()
    activation_values_quanto = activations_quanto[layer_name].cpu().numpy()

    # Calculate percentiles for original model
    min_val_orig = np.min(activation_values_original)
    max_val_orig = np.max(activation_values_original)
    p1_orig = np.percentile(activation_values_original, 1)
    p99_orig = np.percentile(activation_values_original, 99)
    p25_orig = np.percentile(activation_values_original, 25)
    p75_orig = np.percentile(activation_values_original, 75)

    # Calculate percentiles for quantized model
    min_val_quanto = np.min(activation_values_quanto)
    max_val_quanto = np.max(activation_values_quanto)
    p1_quanto = np.percentile(activation_values_quanto, 1)
    p99_quanto = np.percentile(activation_values_quanto, 99)
    p25_quanto = np.percentile(activation_values_quanto, 25)
    p75_quanto = np.percentile(activation_values_quanto, 75)

    # Plot histogram with log scale for both models
    plt.figure(figsize=(20, 6))

    # Original model histogram
    plt.subplot(1, 2, 1)
    plt.hist(activation_values_original.flatten(), bins=100, color='blue', alpha=0.7, log=True)
    plt.axvline(min_val_orig, color='blue', linestyle='dashed', linewidth=2, label='Min/Max')
    plt.axvline(max_val_orig, color='blue', linestyle='dashed', linewidth=2)
    plt.axvline(p1_orig, color='red', linestyle='dashed', linewidth=2, label='1/99 Percentile')
    plt.axvline(p99_orig, color='red', linestyle='dashed', linewidth=2)
    plt.axvline(p25_orig, color='orange', linestyle='dashed', linewidth=2, label='25/75 Percentile')
    plt.axvline(p75_orig, color='orange', linestyle='dashed', linewidth=2)
    plt.title(f"Histogram of Activation Values - {layer_name} (Original)")
    plt.xlabel("Activation Value")
    plt.ylabel("Log-Scaled Frequency")
    plt.legend()
    plt.grid(True)

    # Quantized model histogram
    plt.subplot(1, 2, 2)
    plt.hist(activation_values_quanto.flatten(), bins=100, color='blue', alpha=0.7, log=True)
    plt.axvline(min_val_quanto, color='blue', linestyle='dashed', linewidth=2, label='Min/Max')
    plt.axvline(max_val_quanto, color='blue', linestyle='dashed', linewidth=2)
    plt.axvline(p1_quanto, color='red', linestyle='dashed', linewidth=2, label='1/99 Percentile')
    plt.axvline(p99_quanto, color='red', linestyle='dashed', linewidth=2)
    plt.axvline(p25_quanto, color='orange', linestyle='dashed', linewidth=2, label='25/75 Percentile')
    plt.axvline(p75_quanto, color='orange', linestyle='dashed', linewidth=2)
    plt.title(f"Histogram of Activation Values - {layer_name} (Quantized)")
    plt.xlabel("Activation Value")
    plt.ylabel("Log-Scaled Frequency")
    plt.legend()
    plt.grid(True)

    # Save the histograms
    os.makedirs('plots/activations_quanto', exist_ok=True)
    plt.savefig(f"plots/activations_quanto/{layer_name}_histogram.png")
    plt.close()

    # Plot activations in original order with percentile bands for both models
    plt.figure(figsize=(20, 6))

    # Original model activations
    plt.subplot(1, 2, 1)
    plt.plot(activation_values_original.flatten(), color='blue', alpha=0.7, label="Activations")
    plt.fill_between(range(len(activation_values_original.flatten())), p25_orig, p75_orig, color='orange', alpha=0.5, label="25/75 Percentile")
    plt.fill_between(range(len(activation_values_original.flatten())), p1_orig, p99_orig, color='red', alpha=0.3, label="1/99 Percentile")
    plt.plot(np.full_like(activation_values_original.flatten(), min_val_orig), color='blue', linestyle='dashed', linewidth=2)
    plt.plot(np.full_like(activation_values_original.flatten(), max_val_orig), color='blue', linestyle='dashed', linewidth=2, label="Min/Max")
    plt.title(f"Activation Values in Order - {layer_name} (Original)")
    plt.xlabel("Activation Index")
    plt.ylabel("Activation Value")
    plt.legend(loc="upper left")
    plt.grid(True)

    # Quantized model activations
    plt.subplot(1, 2, 2)
    plt.plot(activation_values_quanto.flatten(), color='blue', alpha=0.7, label="Activations")
    plt.fill_between(range(len(activation_values_quanto.flatten())), p25_quanto, p75_quanto, color='orange', alpha=0.5, label="25/75 Percentile")
    plt.fill_between(range(len(activation_values_quanto.flatten())), p1_quanto, p99_quanto, color='red', alpha=0.3, label="1/99 Percentile")
    plt.plot(np.full_like(activation_values_quanto.flatten(), min_val_quanto), color='blue', linestyle='dashed', linewidth=2)
    plt.plot(np.full_like(activation_values_quanto.flatten(), max_val_quanto), color='blue', linestyle='dashed', linewidth=2, label="Min/Max")
    plt.title(f"Activation Values in Order - {layer_name} (Quantized)")
    plt.xlabel("Activation Index")
    plt.ylabel("Activation Value")
    plt.legend(loc="upper left")
    plt.grid(True)

    # Save the ordered activations
    plt.savefig(f"plots/activations_quanto/{layer_name}_ordered.png")
    plt.close()

# Load the GPT-2 model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "TinyLlama/TinyLlama_v1.1"
print(f"Loading model: {model_name}...")
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")

# Example input
input_text = "The quick brown fox"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)

# Extract activations for the original model
activations_original = {}
print("Registering hooks for the original model...")
for name, module in model.named_modules():
    if 'mlp' in name or 'attn' in name:
        module.register_forward_hook(get_activation(name, activations_original))

# Perform a forward pass
print("Performing forward pass for the original model...")
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits

# Quantize the model using QUANTO
print("Quantizing the model using QUANTO...")
quantize(model, weights=qint8, activations=qint8)
freeze(model)

# Extract activations for the quantized model
activations_quanto = {}
print("Registering hooks for the quantized model...")
for name, module in model.named_modules():
    if 'mlp' in name or 'attn' in name:
        module.register_forward_hook(get_activation(name, activations_quanto))

# Perform a forward pass with the quantized model
print("Performing forward pass for the quantized model...")
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits

# Plot activations for both the original and quantized models
print("Plotting activations for both models...")
for layer_name in activations_original.keys():
    plot_activations_pair(layer_name, activations_original, activations_quanto)

print("Plots saved for original and quantized models.")

Loading model: TinyLlama/TinyLlama_v1.1...
Registering hooks for the original model...
Performing forward pass for the original model...
Quantizing the model using QUANTO...
Registering hooks for the quantized model...
Performing forward pass for the quantized model...
Plotting activations for both models...
Plots saved for original and quantized models.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
import os
import numpy as np
from optimum.quanto import quantize, freeze, qint8, qfloat8, safe_save

# Helper functions to register hooks and extract activations
def get_activation(name, activations):
    def hook(model, input, output):
        if isinstance(output, tuple):
            output = output[0]
        activations[name] = output.detach()
    return hook

def plot_activations_pair(layer_name, activations_original, activations_quanto):
    activation_values_original = activations_original[layer_name].cpu().numpy()
    activation_values_quanto = activations_quanto[layer_name].cpu().numpy()

    # Calculate percentiles for original model
    min_val_orig = np.min(activation_values_original)
    max_val_orig = np.max(activation_values_original)
    p1_orig = np.percentile(activation_values_original, 1)
    p99_orig = np.percentile(activation_values_original, 99)
    p25_orig = np.percentile(activation_values_original, 25)
    p75_orig = np.percentile(activation_values_original, 75)

    # Calculate percentiles for quantized model
    min_val_quanto = np.min(activation_values_quanto)
    max_val_quanto = np.max(activation_values_quanto)
    p1_quanto = np.percentile(activation_values_quanto, 1)
    p99_quanto = np.percentile(activation_values_quanto, 99)
    p25_quanto = np.percentile(activation_values_quanto, 25)
    p75_quanto = np.percentile(activation_values_quanto, 75)

    # Plot histogram with log scale for both models
    plt.figure(figsize=(20, 6))

    # Original model histogram
    plt.subplot(1, 2, 1)
    plt.hist(activation_values_original.flatten(), bins=100, color='blue', alpha=0.7, log=True)
    plt.axvline(min_val_orig, color='blue', linestyle='dashed', linewidth=2, label='Min/Max')
    plt.axvline(max_val_orig, color='blue', linestyle='dashed', linewidth=2)
    plt.axvline(p1_orig, color='red', linestyle='dashed', linewidth=2, label='1/99 Percentile')
    plt.axvline(p99_orig, color='red', linestyle='dashed', linewidth=2)
    plt.axvline(p25_orig, color='orange', linestyle='dashed', linewidth=2, label='25/75 Percentile')
    plt.axvline(p75_orig, color='orange', linestyle='dashed', linewidth=2)
    plt.title(f"Histogram of Activation Values - {layer_name} (Original)")
    plt.xlabel("Activation Value")
    plt.ylabel("Log-Scaled Frequency")
    plt.legend()
    plt.grid(True)

    # Quantized model histogram
    plt.subplot(1, 2, 2)
    plt.hist(activation_values_quanto.flatten(), bins=100, color='blue', alpha=0.7, log=True)
    plt.axvline(min_val_quanto, color='blue', linestyle='dashed', linewidth=2, label='Min/Max')
    plt.axvline(max_val_quanto, color='blue', linestyle='dashed', linewidth=2)
    plt.axvline(p1_quanto, color='red', linestyle='dashed', linewidth=2, label='1/99 Percentile')
    plt.axvline(p99_quanto, color='red', linestyle='dashed', linewidth=2)
    plt.axvline(p25_quanto, color='orange', linestyle='dashed', linewidth=2, label='25/75 Percentile')
    plt.axvline(p75_quanto, color='orange', linestyle='dashed', linewidth=2)
    plt.title(f"Histogram of Activation Values - {layer_name} (Quantized)")
    plt.xlabel("Activation Value")
    plt.ylabel("Log-Scaled Frequency")
    plt.legend()
    plt.grid(True)

    # Save the histograms
    os.makedirs('plots/activations_quanto', exist_ok=True)
    plt.savefig(f"plots/activations_quanto/{layer_name}_histogram.png")
    plt.close()

    # Plot activations in original order with percentile bands for both models
    plt.figure(figsize=(20, 6))

    # Original model activations
    plt.subplot(1, 2, 1)
    plt.plot(activation_values_original.flatten(), color='blue', alpha=0.7, label="Activations")
    plt.fill_between(range(len(activation_values_original.flatten())), p25_orig, p75_orig, color='orange', alpha=0.5, label="25/75 Percentile")
    plt.fill_between(range(len(activation_values_original.flatten())), p1_orig, p99_orig, color='red', alpha=0.3, label="1/99 Percentile")
    plt.plot(np.full_like(activation_values_original.flatten(), min_val_orig), color='blue', linestyle='dashed', linewidth=2)
    plt.plot(np.full_like(activation_values_original.flatten(), max_val_orig), color='blue', linestyle='dashed', linewidth=2, label="Min/Max")
    plt.title(f"Activation Values in Order - {layer_name} (Original)")
    plt.xlabel("Activation Index")
    plt.ylabel("Activation Value")
    plt.legend(loc="upper left")
    plt.grid(True)

    # Quantized model activations
    plt.subplot(1, 2, 2)
    plt.plot(activation_values_quanto.flatten(), color='blue', alpha=0.7, label="Activations")
    plt.fill_between(range(len(activation_values_quanto.flatten())), p25_quanto, p75_quanto, color='orange', alpha=0.5, label="25/75 Percentile")
    plt.fill_between(range(len(activation_values_quanto.flatten())), p1_quanto, p99_quanto, color='red', alpha=0.3, label="1/99 Percentile")
    plt.plot(np.full_like(activation_values_quanto.flatten(), min_val_quanto), color='blue', linestyle='dashed', linewidth=2)
    plt.plot(np.full_like(activation_values_quanto.flatten(), max_val_quanto), color='blue', linestyle='dashed', linewidth=2, label="Min/Max")
    plt.title(f"Activation Values in Order - {layer_name} (Quantized)")
    plt.xlabel("Activation Index")
    plt.ylabel("Activation Value")
    plt.legend(loc="upper left")
    plt.grid(True)

    # Save the ordered activations
    plt.savefig(f"plots/activations_quanto/{layer_name}_ordered.png")
    plt.close()

# Load the GPT-2 model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "TinyLlama/TinyLlama_v1.1"
print(f"Loading model: {model_name}...")
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")

# Example input
input_text = "The quick brown fox"
input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(device)

# Extract activations for the original model
activations_original = {}
print("Registering hooks for the original model...")
for name, module in model.named_modules():
    if 'mlp' in name or 'attn' in name:
        module.register_forward_hook(get_activation(name, activations_original))

# Perform a forward pass
print("Performing forward pass for the original model...")
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits

# Quantize the model using QUANTO
print("Quantizing the model using QUANTO...")
quantize(model, weights=qint8, activations=qint8)
freeze(model)

# Extract activations for the quantized model
activations_quanto = {}
print("Registering hooks for the quantized model...")
for name, module in model.named_modules():
    if 'mlp' in name or 'attn' in name:
        module.register_forward_hook(get_activation(name, activations_quanto))

# Perform a forward pass with the quantized model
print("Performing forward pass for the quantized model...")
with torch.no_grad():
    outputs = model(input_ids)
    logits = outputs.logits

# Plot activations for both the original and quantized models
print("Plotting activations for both models...")
for layer_name in activations_original.keys():
    plot_activations_pair(layer_name, activations_original, activations_quanto)

print("Plots saved for original and quantized models.")